# EXP24: 连续亏损后行为分析

1. 连续亏损后继续交易 vs 暂停交易的对比
2. 不同冷却策略的效果
3. 连续亏损后反弹概率

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, C12_KWARGS
import pandas as pd
import numpy as np

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}

baseline = run_variant(data, "Baseline", **LIVE_KWARGS)
trades = baseline['_trades']
pnls = [t.pnl for t in trades]
print(f"Total: {len(trades)} trades")

In [ ]:
# Part 1: 连续亏损后 N 笔的平均表现
print("=== Performance After N Consecutive Losses ===")
for streak_len in [2, 3, 4, 5, 6, 7]:
    after_streak = []
    consec = 0
    for i, pnl in enumerate(pnls):
        if pnl < 0:
            consec += 1
        else:
            if consec >= streak_len and i < len(pnls):
                # next N trades after streak ends
                for j in range(i, min(i+5, len(pnls))):
                    after_streak.append(pnls[j])
            consec = 0
    
    if len(after_streak) > 0:
        avg = np.mean(after_streak)
        wr = sum(1 for p in after_streak if p > 0) / len(after_streak) * 100
        print(f"  After {streak_len}+ losses: {len(after_streak)} trades, "
              f"avg=${avg:.2f}, WR={wr:.1f}%, total=${sum(after_streak):.0f}")
    else:
        print(f"  After {streak_len}+ losses: no occurrences")

In [ ]:
# Part 2: 模拟跳过连续亏损后的交易
print("\n=== Simulated Skip After Consecutive Losses ===")
for skip_after in [3, 4, 5]:
    for skip_n in [1, 2, 3, 5]:
        new_pnls = []
        consec = 0
        skip_remaining = 0
        skipped = 0
        for pnl in pnls:
            if skip_remaining > 0:
                skip_remaining -= 1
                skipped += 1
                continue
            new_pnls.append(pnl)
            if pnl < 0:
                consec += 1
                if consec >= skip_after:
                    skip_remaining = skip_n
            else:
                consec = 0
        
        total = sum(new_pnls)
        orig_total = sum(pnls)
        delta = total - orig_total
        print(f"  Skip {skip_n} after {skip_after} losses: "
              f"{len(new_pnls)} trades (skipped {skipped}), "
              f"PnL=${total:.0f} (D={delta:+.0f})")

In [ ]:
# Part 3: 连续亏损分布
streaks = []
consec = 0
for pnl in pnls:
    if pnl < 0:
        consec += 1
    else:
        if consec > 0:
            streaks.append(consec)
        consec = 0
if consec > 0:
    streaks.append(consec)

print(f"\n=== Loss Streak Distribution ({len(streaks)} streaks) ===")
for length in sorted(set(streaks)):
    count = streaks.count(length)
    print(f"  {length} consecutive: {count} times ({count/len(streaks)*100:.1f}%)")

print(f"\nMax streak: {max(streaks)}")
print(f"Avg streak: {np.mean(streaks):.1f}")
print(f"P90 streak: {np.percentile(streaks, 90):.0f}")
print(f"P99 streak: {np.percentile(streaks, 99):.0f}")